# Yield prediction
Predict maize yield in kg/acre from the audited Kenya survey data. The audit is complete; this notebook focuses on modelling.

Drought, flood and realized input use describe the growing season, so this is a model using season information, not a forecast available at planting time. Associations and feature importance do not establish causal effects.

## Approach
- Keep the corrected notebook's eligible rows so model comparisons use the same farms.
- Compare total fertilizer amounts with rates per acre, and test a small set of additional survey features.
- Use only the previous calendar year's district yield as a historical feature.
- Compare Ridge and two small gradient-boosting models on 2018 and 2019, always training on earlier years. Report simple median benchmarks as well.
- Select the model with the lowest mean validation MAE across the two years, then fit on 2016–2019 and evaluate once on 2020.

The 2020 results and earlier exploratory analysis have already been viewed, so this is a previously inspected holdout. Later data would be needed for independent confirmation. Performance in two validation years does not establish reliability in every future season.


In [1]:
import pandas as pd
import numpy as np

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [2]:
df = pd.read_csv('data/cleaned/kenya_cleaned.csv', low_memory=False)
TARGET = 'yield_kg_pa'
print('Dataset shape:', df.shape)
print(df.groupby('year')[TARGET].agg(['count', 'mean', 'median']).round(1))


Dataset shape: (23674, 144)
      count    mean  median
year                       
2016   3876  1142.0  1060.5
2017   3991  1419.4  1336.8
2018   4245  1288.6  1180.3
2019   5372  1265.3  1174.7
2020   6190  1262.8  1200.0


## Prepare features
Use the existing complete-case rule for measured inputs, and retain missing district history. Imputation and encoding are fitted separately inside each training fold. Historical medians are joined by district and year, never shifted across farm rows.

The rates and extended feature sets exclude year: a linear trend need not continue into the next season. The existing feature set retains numeric year for comparison. Cow ownership combines the audited count and binary fields; missing counts remain missing unless the binary field supplies an answer.


In [3]:
CATEGORICAL_COLS = ['district_clean', 'fertility', 'drought', 'flood', 'seed_class']
EXISTING_NUMERIC = ['year', 'dap_kg', 'urea_kg', 'npk_kg', 'can_kg', 'plot_acres']
RATE_COLS = ['dap_kg_pa', 'urea_kg_pa', 'npk_kg_pa', 'can_kg_pa',
             'plot_acres', 'lag_district_year_yield']
FEATURE_SETS = {
    'existing': EXISTING_NUMERIC + ['lag_district_year_yield'],
    'rates': RATE_COLS,
    'extended': RATE_COLS + ['lime_kg_pa', 'compost', 'intercrop', 'weed', 'owns_cow']
}

model_df = df.copy()
model_df['owns_cow'] = np.where(
    model_df['cows'].notna(),
    model_df['cows'].gt(0).astype(float),
    model_df['cows_binary']
)
for col in CATEGORICAL_COLS:
    model_df[col] = model_df[col].fillna('missing').astype(str)

numeric_cols = sorted(set(EXISTING_NUMERIC + RATE_COLS + FEATURE_SETS['extended'])
                      - {'lag_district_year_yield'})
for col in numeric_cols + [TARGET]:
    model_df[col] = pd.to_numeric(model_df[col], errors='coerce')

district_history = (
    model_df.groupby(['district_clean', 'year'], as_index=False)[TARGET].median()
    .rename(columns={TARGET: 'lag_district_year_yield'})
)
district_history['year'] = district_history['year'] + 1
model_df = model_df.merge(
    district_history, on=['district_clean', 'year'], how='left', validate='many_to_one'
)
model_df = model_df.dropna(subset=EXISTING_NUMERIC + [TARGET]).copy()
print('Eligible rows by year:')
print(model_df.groupby('year').size())


Eligible rows by year:
year
2016    3876
2017    3991
2018    4245
2019    5362
2020    6047
dtype: int64


In [4]:
# Each call creates independent preprocessing fitted only on its training data.
def make_model(numeric_cols, estimator):
    preprocessor = ColumnTransformer([
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
            ('scaler', StandardScaler())
        ]), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False),
         CATEGORICAL_COLS)
    ])
    return Pipeline([('preprocess', preprocessor), ('model', clone(estimator))])

models = {
    'Ridge': Ridge(alpha=100),
    'Boosting small': HistGradientBoostingRegressor(
        max_iter=150, max_leaf_nodes=7, min_samples_leaf=50,
        l2_regularization=10, learning_rate=0.05,
        early_stopping=False, random_state=42
    ),
    'Boosting medium': HistGradientBoostingRegressor(
        max_iter=150, max_leaf_nodes=15, min_samples_leaf=50,
        l2_regularization=10, learning_rate=0.05,
        early_stopping=False, random_state=42
    )
}


## Compare on earlier years
Train through 2017 to predict 2018, and through 2018 to predict 2019. Give each validation year equal weight in model selection. MAE is the primary metric; RMSE and R² show complementary aspects of error. The training median and previous-year district median establish whether the models add useful predictive information.


In [5]:
validation_rows = []
for validation_year in [2018, 2019]:
    train = model_df[model_df['year'] < validation_year]
    valid = model_df[model_df['year'] == validation_year]
    y_valid = valid[TARGET]

    predictions = {
        ('baseline', 'Training median'): np.repeat(train[TARGET].median(), len(valid)),
        ('baseline', 'Previous district median'):
            valid['lag_district_year_yield'].fillna(train[TARGET].median())
    }
    for feature_name, numeric_cols in FEATURE_SETS.items():
        features = numeric_cols + CATEGORICAL_COLS
        for model_name, estimator in models.items():
            pipeline = make_model(numeric_cols, estimator)
            pipeline.fit(train[features], train[TARGET])
            predictions[(feature_name, model_name)] = pipeline.predict(valid[features])

    for (feature_name, model_name), pred in predictions.items():
        validation_rows.append({
            'features': feature_name, 'model': model_name, 'year': validation_year,
            'MAE': mean_absolute_error(y_valid, pred),
            'RMSE': np.sqrt(mean_squared_error(y_valid, pred)),
            'R2': r2_score(y_valid, pred)
        })

validation_results = pd.DataFrame(validation_rows)
validation_summary = (
    validation_results.groupby(['features', 'model'])[['MAE', 'RMSE', 'R2']]
    .mean().sort_values('MAE')
)
print('Mean validation scores (equal weight per year):')
print(validation_summary.round(3).to_string())
print('\nMAE by validation year:')
print(validation_results.pivot(index=['features', 'model'], columns='year', values='MAE')
      .round(2).to_string())


Mean validation scores (equal weight per year):
                                       MAE     RMSE     R2
features model                                            
extended Boosting medium           512.418  653.829  0.111
rates    Ridge                     512.715  648.362  0.125
extended Boosting small            514.312  652.828  0.112
existing Boosting small            514.568  654.500  0.107
rates    Boosting medium           515.704  654.417  0.109
existing Boosting medium           516.466  657.989  0.099
rates    Boosting small            519.338  656.502  0.101
extended Ridge                     526.488  665.551  0.072
baseline Training median           538.169  701.345 -0.018
         Previous district median  539.795  691.679  0.007
existing Ridge                     554.334  701.733 -0.036

MAE by validation year:
year                                 2018    2019
features model                                   
baseline Previous district median  539.90  539.69
         T

## Evaluate the selected model on 2020
Choose the learned model using validation MAE only, then compare it with both benchmarks on the same holdout farms. Keep this comparison even if a benchmark wins. Do not revise the model based on this holdout result.


In [6]:
candidate_scores = validation_summary.drop(index='baseline', level='features')
best_features, best_model_name = candidate_scores.index[0]
print('Selected:', best_features, '/', best_model_name)

train_df = model_df[model_df['year'] < 2020].copy()
test_df = model_df[model_df['year'] == 2020].copy()
numeric_cols = FEATURE_SETS[best_features]
features = numeric_cols + CATEGORICAL_COLS
X_train, y_train = train_df[features], train_df[TARGET]
X_test, y_test = test_df[features], test_df[TARGET]

best_pipeline = make_model(numeric_cols, models[best_model_name])
best_pipeline.fit(X_train, y_train)
best_preds = best_pipeline.predict(X_test)

holdout_predictions = {
    'Training median': np.repeat(y_train.median(), len(y_test)),
    'Previous district median': test_df['lag_district_year_yield'].fillna(y_train.median()),
    best_model_name + ' (' + best_features + ')': best_preds
}
holdout_rows = []
for name, pred in holdout_predictions.items():
    holdout_rows.append({
        'model': name, 'MAE': mean_absolute_error(y_test, pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, pred)), 'R2': r2_score(y_test, pred)
    })
holdout_results = pd.DataFrame(holdout_rows).set_index('model')
print(f'Training rows: {len(train_df)}; holdout rows: {len(test_df)}')
print(holdout_results.round(3).to_string())


Selected: extended / Boosting medium
Training rows: 17474; holdout rows: 6047
                                MAE     RMSE     R2
model                                              
Training median             497.368  647.689 -0.016
Previous district median    531.591  686.929 -0.143
Boosting medium (extended)  473.397  608.119  0.104


In [7]:
# Locate systematic errors instead of relying on a single overall score.
errors = test_df[['district_clean', 'seed_class', TARGET]].copy()
errors['prediction'] = best_preds
errors['absolute_error'] = (errors[TARGET] - errors['prediction']).abs()
errors['bias'] = errors['prediction'] - errors[TARGET]
print('District errors (positive bias means overprediction):')
print(errors.groupby('district_clean').agg(
    n=(TARGET, 'size'), MAE=('absolute_error', 'mean'), bias=('bias', 'mean')
).sort_values('MAE', ascending=False).round(1).to_string())


District errors (positive bias means overprediction):
                  n     MAE   bias
district_clean                    
njoro            33  1087.9 -840.6
imenti           55   812.1  730.6
saboti           88   746.0 -556.7
olkalou          50   682.5 -225.0
manyatta         55   640.0  582.5
ndalu           188   631.9  495.0
mumias          156   621.1  329.7
belgut          124   571.4  377.1
sagana           93   562.3  -95.6
keiyo           118   560.2 -305.2
tinderet         73   558.7  361.9
masaba           88   543.0 -352.6
rongo            84   529.1  -57.8
greenshamba     105   526.8  265.6
lugari          220   520.3  241.2
webuye          132   515.5   82.0
kapsabet         51   513.5 -274.6
suneka           87   511.3  183.9
kabiyet         101   492.9   52.0
kipkelion        37   488.7  335.0
kakamega_south  239   487.6  129.3
serabo          142   487.0  373.8
busia           207   469.6  -53.2
kabondo          70   464.2 -163.8
kimilili        186   462.6 -101.2
k

## What the results support

The selected extended boosting model has 2020 MAE **473.40 kg/acre**, RMSE **608.12 kg/acre**, and R² **0.104**. Its MAE is only **4.8% lower** than the training-median benchmark (497.37 kg/acre). It does **not** improve on the earlier corrected linear regression's MAE of 465.63 kg/acre on the same holdout rows.

The validation choice is also close: extended boosting averages 512.42 kg/acre MAE, versus 512.71 for Ridge with per-acre inputs. This small difference is not convincing evidence that the more complex model is better. Performance in 2018 is particularly weak, and district-level errors reveal large systematic over- and underprediction.

These results do not yet support reliable farm-level forecasts. They show limited predictive improvement from the tested features and models; they do not prove that the dataset has an unavoidable accuracy ceiling. Further work should examine the year-to-year stability of the district and input relationships using development years, and establish the error level the intended use can tolerate. Any further model selection needs to remain within development-year validation. Since 2020 has already been inspected repeatedly, independent confirmation needs a new season of data.


# District-level yield prediction

**Target:** the arithmetic mean of surveyed farms' `yield_kg_pa` in each district and year. This describes the surveyed farms, not necessarily all farms in a district. It is not an area-weighted yield or total district production.

Start from the audited dataset, retaining farms with a recorded district, year and target. Aggregate to one row per district-year. Report farm counts, but give each district equal weight when fitting and evaluating: the objective is one prediction per district.

Fertilizer rates and plot size are averaged over recorded values. Seed and fertility shares use recorded, relevant responses; drought and flood shares use recorded responses. Coverage features distinguish low reporting from a low prevalence. Current-season inputs and stress reports mean this remains a prediction using season information, not a planting-time forecast.

With only 180 district-years, use Ridge regression with three regularization strengths. Validate on 2018 and 2019 using earlier years only. Compare with the previous year's mean across districts and each district's own previous-year yield. Select by mean validation MAE, allowing a benchmark to win. Keep 2020 for the final comparison; it remains a previously inspected holdout.


In [8]:
district_source = df.dropna(subset=['district_clean', 'year', TARGET]).copy()

# Missing survey answers stay missing, rather than becoming zero.
district_source['hybrid_share'] = (
    district_source['seed_class'].eq('named hybrid').astype(float)
    .where(district_source['seed_class'].notna())
)
district_source['mixed_seed_share'] = (
    district_source['seed_class'].eq('mixed').astype(float)
    .where(district_source['seed_class'].notna())
)
valid_fertility = district_source['fertility'].isin(['better', 'same', 'worse'])
for label in ['better', 'worse']:
    district_source['fertility_' + label + '_share'] = (
        district_source['fertility'].eq(label).astype(float).where(valid_fertility)
    )

DISTRICT_INPUTS = [
    'dap_kg_pa', 'urea_kg_pa', 'npk_kg_pa', 'can_kg_pa', 'plot_acres',
    'hybrid_share', 'mixed_seed_share', 'fertility_better_share',
    'fertility_worse_share', 'drought', 'flood'
]
for col in DISTRICT_INPUTS:
    district_source[col] = pd.to_numeric(district_source[col], errors='coerce')

district_groups = district_source.groupby(['district_clean', 'year'])
district_df = district_groups[DISTRICT_INPUTS].mean()
district_df['mean_yield'] = district_groups[TARGET].mean()
district_df['n_farms'] = district_groups.size()

# Both fertility shares have the same response coverage.
COVERAGE_INPUTS = [c for c in DISTRICT_INPUTS if c not in
                   ['mixed_seed_share', 'fertility_worse_share']]
for col in COVERAGE_INPUTS:
    district_df[col + '_coverage'] = district_groups[col].count() / district_df['n_farms']
district_df = district_df.reset_index()

# Join the exact previous calendar year; never use current-year target means.
district_lag = district_df[['district_clean', 'year', 'mean_yield']].copy()
district_lag['year'] += 1
district_lag = district_lag.rename(columns={'mean_yield': 'previous_yield'})
district_df = district_df.merge(
    district_lag, on=['district_clean', 'year'], how='left', validate='one_to_one'
)
DISTRICT_NUMERIC = (DISTRICT_INPUTS + [c + '_coverage' for c in COVERAGE_INPUTS]
                    + ['previous_yield'])
DISTRICT_FEATURES = DISTRICT_NUMERIC + ['district_clean']

print('District-year rows:', len(district_df))
print(district_df.groupby('year').agg(
    districts=('district_clean', 'size'), surveyed_farms=('n_farms', 'sum'),
    median_farms_per_district=('n_farms', 'median')
).to_string())


District-year rows: 180
      districts  surveyed_farms  median_farms_per_district
year                                                      
2016         26            3876                      144.5
2017         30            3934                      121.0
2018         35            4245                      112.0
2019         39            5372                      124.0
2020         50            6190                      111.5


In [9]:
def make_district_model(alpha):
    preprocessing = ColumnTransformer([
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
            ('scaler', StandardScaler())
        ]), DISTRICT_NUMERIC),
        ('district', OneHotEncoder(handle_unknown='ignore', sparse_output=False),
         ['district_clean'])
    ])
    return Pipeline([('preprocess', preprocessing), ('model', Ridge(alpha=alpha))])

district_alphas = [1, 10, 100]
district_validation_rows = []
for validation_year in [2018, 2019]:
    district_train = district_df[district_df['year'] < validation_year]
    district_valid = district_df[district_df['year'] == validation_year]
    previous_mean = district_train.loc[
        district_train['year'] == validation_year - 1, 'mean_yield'
    ].mean()
    district_predictions = {
        'Previous overall mean': np.repeat(previous_mean, len(district_valid)),
        'Previous district mean': district_valid['previous_yield'].fillna(previous_mean)
    }
    for alpha in district_alphas:
        district_pipeline = make_district_model(alpha)
        district_pipeline.fit(district_train[DISTRICT_FEATURES], district_train['mean_yield'])
        district_predictions[f'Ridge alpha={alpha}'] = district_pipeline.predict(
            district_valid[DISTRICT_FEATURES]
        )
    for model_name, pred in district_predictions.items():
        district_validation_rows.append({
            'model': model_name, 'year': validation_year,
            'districts': len(district_valid),
            'MAE': mean_absolute_error(district_valid['mean_yield'], pred),
            'RMSE': np.sqrt(mean_squared_error(district_valid['mean_yield'], pred)),
            'R2': r2_score(district_valid['mean_yield'], pred)
        })

district_validation = pd.DataFrame(district_validation_rows)
district_summary = (
    district_validation.groupby('model')[['MAE', 'RMSE', 'R2']]
    .mean().sort_values('MAE')
)
district_selected = district_summary.index[0]
print('Mean validation metrics (equal weight per year):')
print(district_summary.round(3).to_string())
print('Validation metrics by year:')
print(district_validation.round(3).to_string(index=False))
print('Selected using validation only:', district_selected)


Mean validation metrics (equal weight per year):
                            MAE     RMSE     R2
model                                          
Ridge alpha=100         276.963  334.400  0.001
Previous district mean  284.257  367.199 -0.214
Previous overall mean   305.830  383.490 -0.266
Ridge alpha=10          328.697  395.956 -0.734
Ridge alpha=1           332.828  397.199 -0.729
Validation metrics by year:
                 model  year  districts     MAE    RMSE     R2
 Previous overall mean  2018         35 269.894 318.923 -0.531
Previous district mean  2018         35 259.424 325.491 -0.595
         Ridge alpha=1  2018         35 370.711 431.307 -1.800
        Ridge alpha=10  2018         35 377.238 433.446 -1.828
       Ridge alpha=100  2018         35 256.301 293.526 -0.297
 Previous overall mean  2019         39 341.767 448.056 -0.000
Previous district mean  2019         39 309.090 408.907  0.167
         Ridge alpha=1  2019         39 294.945 363.091  0.343
        Ridge alpha=

In [10]:
# Evaluate only the selected Ridge configuration plus the fixed benchmarks.
district_train = district_df[district_df['year'] < 2020].copy()
district_test = district_df[district_df['year'] == 2020].copy()
previous_mean = district_train.loc[district_train['year'] == 2019, 'mean_yield'].mean()
district_holdout_predictions = {
    'Previous overall mean': np.repeat(previous_mean, len(district_test)),
    'Previous district mean': district_test['previous_yield'].fillna(previous_mean)
}
if district_selected.startswith('Ridge'):
    selected_alpha = int(district_selected.split('=')[1])
    district_model = make_district_model(selected_alpha)
    district_model.fit(district_train[DISTRICT_FEATURES], district_train['mean_yield'])
    district_holdout_predictions[district_selected] = district_model.predict(
        district_test[DISTRICT_FEATURES]
    )

district_holdout_rows = []
for model_name, pred in district_holdout_predictions.items():
    district_holdout_rows.append({
        'model': model_name,
        'MAE': mean_absolute_error(district_test['mean_yield'], pred),
        'RMSE': np.sqrt(mean_squared_error(district_test['mean_yield'], pred)),
        'R2': r2_score(district_test['mean_yield'], pred)
    })
district_holdout_results = pd.DataFrame(district_holdout_rows).set_index('model')
print('2020 district-level metrics:')
print(district_holdout_results.round(3).to_string())
print('Selected model:', district_selected)


2020 district-level metrics:
                            MAE     RMSE     R2
model                                          
Previous overall mean   233.516  310.305 -0.001
Previous district mean  300.590  383.514 -0.530
Ridge alpha=100         202.215  275.058  0.213
Selected model: Ridge alpha=100


In [11]:
district_errors = district_test[
    ['district_clean', 'n_farms', 'mean_yield', 'previous_yield']
].copy()
district_errors['prediction'] = np.asarray(district_holdout_predictions[district_selected])
district_errors['absolute_error'] = (
    district_errors['prediction'] - district_errors['mean_yield']
).abs()
district_errors['bias'] = district_errors['prediction'] - district_errors['mean_yield']
district_errors['has_previous_year'] = district_errors['previous_yield'].notna()
print('Errors by availability of previous-year district yield:')
print(district_errors.groupby('has_previous_year').agg(
    districts=('district_clean', 'size'), MAE=('absolute_error', 'mean'),
    bias=('bias', 'mean')
).round(2).to_string())
print('Predictions and errors for each district:')
print(district_errors.sort_values('absolute_error', ascending=False)
      .round(2).to_string(index=False))


Errors by availability of previous-year district yield:
                   districts     MAE    bias
has_previous_year                           
False                     11  329.24  237.56
True                      39  166.39   42.34
Predictions and errors for each district:
district_clean  n_farms  mean_yield  previous_yield  prediction  absolute_error    bias  has_previous_year
        imenti       56      479.36             NaN     1439.98          960.63  960.63              False
      manyatta       57      657.11             NaN     1522.45          865.34  865.34              False
   greenshamba      107      851.27             NaN     1318.74          467.47  467.47              False
         ndalu      194     1342.14         2029.00     1772.22          430.07  430.07               True
          teso      213      999.84         1239.23     1413.66          413.82  413.82               True
      kapsabet       51     1590.07         1356.76     1176.31          413.77 

## Interpreting district-level results

MAE now measures errors in **district sample means**, not individual farms. A lower MAE than the farm-level model is expected from averaging and is not evidence of better farm predictions. Compare district models with district benchmarks on the same rows.

Smaller district samples have less precise observed means, and changing surveyed farms can change a district's measured yield without a corresponding change in its whole farming population. The reported metrics give each district equal weight. Separate results for districts lacking previous-year history reveal how the model behaves where historical yield is unavailable.


### Results

Validation selected **Ridge with alpha=100**. On the 50 districts observed in 2020, MAE is **202.22 kg/acre**, RMSE is **275.06 kg/acre**, and R² is **0.213**. MAE is about **13.4% lower** than the previous year's overall district mean benchmark (233.52 kg/acre), and lower than the previous district mean benchmark (300.59 kg/acre).

The 39 districts with previous-year observations have MAE **166.39 kg/acre**; the 11 without them have MAE **329.24 kg/acre**. This is a descriptive comparison: these groups also differ in other ways, so it does not isolate the effect of historical data. Imenti and Manyatta have particularly large overpredictions.

Temporal performance remains inconsistent: the selected model's validation R² is **−0.297 in 2018** and **0.298 in 2019**. The district model beats both benchmarks on 2020, but those validation results limit confidence in performance for a new season.
